# 02 · GC10-DET - Preprocessing & DataLoaders

Note: non-standard folder layout (`1/`, `2/`, ..., `10/`).
Solution: a custom Dataset with a folder mapping + WeightedRandomSampler for the imbalance.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print(f'PyTorch: {torch.__version__}')
device = get_device()
ASSETS = "cv/gc10"


In [ ]:
DATA_DIR   = Path('../../../data/06_gc10/raw')

CLASS_MAP  = {
    '1': 'punching_hole', '2': 'welding_line', '3': 'crescent_gap',
    '4': 'water_spot',    '5': 'oil_spot',      '6': 'silk_spot',
    '7': 'inclusion',     '8': 'rolled_pit',     '9': 'crease',
    '10': 'waist_folding'
}
CLASSES     = list(CLASS_MAP.values())
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42
torch.manual_seed(SEED)
print(f'Classes: {NUM_CLASSES} | {CLASSES}')

## 1. Custom Dataset

In [ ]:
class GC10Dataset(Dataset):
    """GC10-DET: images in numbered folders 1-10."""
    def __init__(self, root: Path, class_map: dict, transform=None,
                 val_split: float = 0.2, split: str = 'train', seed: int = 42):
        self.transform = transform
        self.classes   = list(class_map.values())
        self.class_to_idx = {v: i for i, v in enumerate(self.classes)}
        self.samples   = []  # (img_path, label_idx)

        rng = np.random.RandomState(seed)
        for folder_id, class_name in class_map.items():
            img_dir = root / folder_id
            imgs = sorted(list(img_dir.glob('*.jpg')) +
                          list(img_dir.glob('*.bmp')) +
                          list(img_dir.glob('*.png')))
            idx = self.class_to_idx[class_name]
            # Train/Val split
            n_val  = max(1, int(len(imgs) * val_split))
            perm   = rng.permutation(len(imgs))
            if split == 'train':
                chosen = [imgs[i] for i in perm[n_val:]]
            else:
                chosen = [imgs[i] for i in perm[:n_val]]
            for p in chosen:
                self.samples.append((p, idx))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

print('GC10Dataset defined.')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
train_set = GC10Dataset(DATA_DIR, CLASS_MAP, train_transforms, split='train')
val_set   = GC10Dataset(DATA_DIR, CLASS_MAP, val_transforms,   split='val')

# Counts for WeightedRandomSampler (imbalance handling)
label_counts = Counter([s[1] for s in train_set.samples])
weights = [1.0 / label_counts[label] for _, label in train_set.samples]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

print(f'Train: {len(train_set)} | Val: {len(val_set)}')
print(f'Label distribution (train): {dict(sorted(label_counts.items()))}')

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler,   num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check
images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')  # [32, 3, 224, 224]
print(f'Label distribution in batch: {Counter(labels.tolist())}')

## Summary

| Parameter | Value |
|---|---|
| Imbalance | WeightedRandomSampler |
| Augmentation | Flip + Rotation + ColorJitter |
| Normalize | ImageNet stats |

➡️ **Next step:** `03_gc10_modeling.ipynb`